# AIC2025 ingest and resumable indexing

Run `00_colab_setup.ipynb` first. Supplied keyframes/metadata are the default source. The raw-video FFmpeg path is only a fallback when the benchmark has no keyframe export.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

# These names are populated in notebook 00. Recreate them if this is a new runtime.
DRIVE_ROOT = Path('/content/drive/MyDrive/HCM-AI-Challenge-2026')
REPO_ROOT = Path('/content/HCM-AI-Challenge-2026')
ARTIFACT_ROOT = Path(os.environ.get('ARTIFACT_ROOT', DRIVE_ROOT / 'artifacts'))
OUTPUT_ROOT = Path(os.environ.get('OUTPUT_ROOT', DRIVE_ROOT / 'outputs'))
PROFILE = os.environ.get('HCM_AI_PROFILE', 'cpu')

def run_cli(script, *arguments):
    command = [sys.executable, script, *map(str, arguments)]
    completed = subprocess.run(
        command, cwd=REPO_ROOT, env=os.environ.copy(), text=True,
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=False,
    )
    if completed.returncode:
        raise RuntimeError(f'{command} failed\nSTDOUT:\n{completed.stdout}\nSTDERR:\n{completed.stderr}')
    return json.loads(completed.stdout.strip().splitlines()[-1])

In [ ]:
# Update this once to match the folder shared from Drive.
AIC2025_ROOT = Path(os.environ.get('AIC2025_ROOT', DRIVE_ROOT / 'AIC2025'))
KEYFRAMES_ROOT = AIC2025_ROOT / 'keyframes'
FRAME_METADATA = None  # e.g. AIC2025_ROOT / 'frames.jsonl' when authoritative timestamps exist

if not AIC2025_ROOT.exists():
    raise FileNotFoundError(f'Set AIC2025_ROOT to your mounted benchmark folder: {AIC2025_ROOT}')
print('Top-level entries:', [path.name for path in sorted(AIC2025_ROOT.iterdir())][:20])
if FRAME_METADATA is None and not KEYFRAMES_ROOT.is_dir():
    raise FileNotFoundError(f'Point KEYFRAMES_ROOT at the supplied keyframe tree: {KEYFRAMES_ROOT}')

In [ ]:
# The manifest has stable frame/video/timestamp/image-path fields and is content-addressed on Drive.
manifest_args = ['--artifact-root', ARTIFACT_ROOT]
if FRAME_METADATA is not None:
    manifest_args += ['--frame-metadata', FRAME_METADATA]
else:
    manifest_args += ['--keyframes-root', KEYFRAMES_ROOT, '--default-fps', '25']
manifest_info = run_cli('scripts/preprocess_videos.py', *manifest_args)
MANIFEST = Path(manifest_info['artifact']['path'])
print(manifest_info)
print('Manifest:', MANIFEST)

In [ ]:
# SigLIP is the first working baseline. `auto` uses Transformers when available and safely falls back to hash embeddings.
# For paper_gpu, add '--encoder', 'beit3'; a failed optional encoder is recorded as skipped.
visual_args = [
    '--manifest', MANIFEST, '--artifact-root', ARTIFACT_ROOT,
    '--profile', PROFILE, '--encoder', 'siglip', '--provider', 'auto',
    '--batch-size', '32',
]
visual_info = run_cli('scripts/build_visual_index.py', *visual_args)
if 'siglip' not in visual_info['indexes']:
    raise RuntimeError(f'No baseline visual index was created: {visual_info}')
SIGLIP_FINGERPRINT = visual_info['indexes']['siglip']['fingerprint']
print(visual_info)

In [ ]:
# Prefer records supplied by the benchmark/team. Turn RUN_PADDLE_OCR on only for a deliberate bulk OCR pass.
OCR_RECORDS = AIC2025_ROOT / 'ocr_records.jsonl'
ASR_RECORDS = AIC2025_ROOT / 'asr_records.jsonl'
RUN_PADDLE_OCR = False

text_args = ['--manifest', MANIFEST, '--artifact-root', ARTIFACT_ROOT, '--profile', PROFILE]
if OCR_RECORDS.is_file():
    text_args += ['--modality', 'ocr', '--ocr-records', OCR_RECORDS]
elif RUN_PADDLE_OCR:
    text_args += ['--modality', 'ocr', '--run-ocr', '--ocr-batch-size', '16']
if ASR_RECORDS.is_file():
    text_args += ['--modality', 'asr', '--asr-records', ASR_RECORDS]

text_info = run_cli('scripts/build_text_index.py', *text_args) if len(text_args) > 6 else {'modalities': {}, 'skipped': {}}
OCR_FINGERPRINT = text_info.get('modalities', {}).get('ocr', {}).get('index', {}).get('fingerprint')
ASR_FINGERPRINT = text_info.get('modalities', {}).get('asr', {}).get('index', {}).get('fingerprint')
print(text_info)

In [ ]:
# Persist only IDs/paths, never API keys. Notebook 02 reloads this state after a runtime restart.
state = {
    'profile': PROFILE,
    'artifact_root': str(ARTIFACT_ROOT),
    'manifest': str(MANIFEST),
    'visual_indexes': {'siglip': SIGLIP_FINGERPRINT},
    'ocr_index': OCR_FINGERPRINT,
    'asr_index': ASR_FINGERPRINT,
    'aic2025_root': str(AIC2025_ROOT),
}
state_path = ARTIFACT_ROOT / 'colab_state.json'
state_path.write_text(json.dumps(state, ensure_ascii=False, indent=2), encoding='utf-8')
print(state_path)
print(state)

In [ ]:
# Acceptance check: a matching command must reuse Drive vectors instead of re-encoding images.
resume_check = run_cli('scripts/build_visual_index.py', *visual_args)
assert resume_check['indexes']['siglip']['reused'] is True, resume_check
print('Resume verified:', resume_check['indexes']['siglip']['fingerprint'])

In [ ]:
# Inspect benchmark queries (TXT and Excel are both supported; openpyxl is installed by the setup notebook).
from hcm_ai.ingestion import load_aic2025_queries
queries = load_aic2025_queries(AIC2025_ROOT / 'query') if (AIC2025_ROOT / 'query').exists() else []
[(query.query_id, query.task.value, query.text[:100]) for query in queries[:10]]